# PRAGYAAN – Units 3 and 4
Unit 3: dimensionality reduction and clustering. Unit 4: Markov chain and HMM sequence models.
Data: English subset of the Kaggle Multilingual Customer Support Tickets dataset.

In [ ]:
# Upload tickets.parquet and emb.npy (from the zip) to the Colab session first, then run all cells.
!pip -q install xgboost hmmlearn pyarrow
import os; os.makedirs('/content/report_work/fig', exist_ok=True)
!mv -f /content/tickets.parquet /content/emb.npy /content/report_work/ 2>/dev/null; ls /content/report_work

In [ ]:
%%writefile /content/common.py
"""Shared data loading, split and plotting style for the PRAGYAAN report experiments."""
import re
import sys
import time

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

import re as _re
_PH = _re.compile(r"<(?:tel_num|email|url|account_num|order_num|ip|date|name|address|password|username|phone|amount|id)[^>]*>", _re.I)


def clean_text(text, strict=False):
    out = _re.sub(r"\s+", " ", _PH.sub(" ", text or "")).strip()
    return out.lower() if strict else out

W = "/content/report_work"
FIG = f"{W}/fig"
LABELS = ["high", "medium", "low"]
SEED = 42

plt.rcParams.update({
    "font.family": "Liberation Serif", "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.bbox": "tight",
})
C = {"high": "#B23A48", "medium": "#E0A458", "low": "#4E8098", "ink": "#2B2D42", "grey": "#8D99AE"}


def load():
    df = pd.read_parquet(f"{W}/tickets.parquet")
    for c in ["text", "priority", "type", "queue", "answer"]:
        df[c] = np.array(df[c].tolist(), dtype=object)
    emb = np.load(f"{W}/emb.npy")
    df["clean"] = [clean_text(t, strict=True) for t in df["text"]]
    df["words"] = df["text"].str.split().str.len()
    return df, emb


def split(df):
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=0.2, stratify=df["priority"], random_state=SEED)
    return tr, te


def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), max_features=2000, stop_words="english", min_df=2)


def evaluate(name, model, Xtr, ytr, Xte, yte, proba=True):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_s = time.perf_counter() - t0
    pred = model.predict(Xte)
    p, r, f, s = precision_recall_fscore_support(yte, pred, labels=LABELS, zero_division=0)
    out = {"model": name, "accuracy": accuracy_score(yte, pred), "macro_f1": f1_score(yte, pred, average="macro"),
           "weighted_f1": f1_score(yte, pred, average="weighted"), "fit_seconds": fit_s,
           "per_class": {lab: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                         for i, lab in enumerate(LABELS)}}
    if proba and hasattr(model, "predict_proba"):
        try:
            P = model.predict_proba(Xte)
            out["roc_auc_macro"] = roc_auc_score(yte, P, multi_class="ovr", average="macro", labels=list(model.classes_))
        except Exception:
            pass
    elif hasattr(model, "decision_function"):
        D = model.decision_function(Xte)
        out["roc_auc_macro"] = _auc_from_scores(yte, D, list(model.classes_))
    out["pred"] = list(pred)
    return out


def _auc_from_scores(y, D, classes):
    from scipy.special import softmax
    return roc_auc_score(y, softmax(D, axis=1), multi_class="ovr", average="macro", labels=classes)


def words_re(s):
    return re.findall(r"[a-z]+", s.lower())


In [ ]:
"""Unit 3: dimensionality reduction (Truncated SVD / PCA) and clustering (K-Means, agglomerative) of tickets,
plus the unsupervised nearest-neighbour retrieval that powers Find a fix."""
import json

import numpy as np
import pandas as pd
import sys; sys.path.insert(0, '/content')
from common import FIG, SEED, C, W, load, plt, tfidf
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.metrics import (adjusted_rand_score, completeness_score, homogeneity_score, normalized_mutual_info_score,
                             silhouette_score)
from sklearn.preprocessing import normalize

df, emb = load()
R = {}
rng = np.random.default_rng(SEED)

# ------------------------------------------------------------------ dimensionality reduction
vec = tfidf(); X = vec.fit_transform(df["clean"])
svd = TruncatedSVD(n_components=300, random_state=SEED).fit(X)
cum_svd = np.cumsum(svd.explained_variance_ratio_)
R["svd_var_100"] = float(cum_svd[99]); R["svd_var_300"] = float(cum_svd[299])

E = normalize(emb)
pca_full = PCA(random_state=SEED).fit(E)
cum_pca = np.cumsum(pca_full.explained_variance_ratio_)
R["pca_n80"] = int(np.searchsorted(cum_pca, 0.80) + 1); R["pca_n90"] = int(np.searchsorted(cum_pca, 0.90) + 1)
R["pca_var_2"] = float(cum_pca[1]); R["pca_var_50"] = float(cum_pca[49])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(range(1, 301), cum_svd * 100, color=C["ink"])
ax[0].set_xlabel("Number of SVD components"); ax[0].set_ylabel("Cumulative variance explained (%)")
ax[0].set_title("(a) Truncated SVD on TF-IDF (2,000 features)")
ax[1].plot(range(1, 385), cum_pca * 100, color=C["ink"])
for thr, n in ((80, R["pca_n80"]), (90, R["pca_n90"])):
    ax[1].axhline(thr, ls=":", color=C["grey"]); ax[1].axvline(n, ls=":", color=C["grey"])
    ax[1].text(n + 4, thr - 6, f"{thr}% at {n}", fontsize=9)
ax[1].set_xlabel("Number of principal components"); ax[1].set_title("(b) PCA on MiniLM embeddings (384 dimensions)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_2_variance.png"); plt.close()

Z = PCA(n_components=50, random_state=SEED).fit_transform(E)
Zs = normalize(Z)

# ------------------------------------------------------------------ choosing k
ks = list(range(2, 21)); inertia, sil = [], []
sample = rng.choice(len(Zs), 3000, replace=False)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(Zs)
    inertia.append(km.inertia_); sil.append(silhouette_score(Zs[sample], km.labels_[sample], metric="cosine"))
R["k_inertia"] = dict(zip(map(str, ks), map(float, inertia))); R["k_silhouette"] = dict(zip(map(str, ks), map(float, sil)))
best_sil_k = ks[int(np.argmax(sil))]; R["best_sil_k"] = best_sil_k
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(ks, inertia, "o-", color=C["ink"], ms=4); ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia (within-cluster SSE)")
ax[0].set_title("(a) Elbow method"); ax[0].axvline(10, ls=":", color=C["high"])
ax[1].plot(ks, sil, "o-", color=C["ink"], ms=4); ax[1].set_xlabel("k"); ax[1].set_ylabel("Silhouette (cosine)")
ax[1].set_title("(b) Silhouette score"); ax[1].axvline(10, ls=":", color=C["high"])
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_3_choose_k.png"); plt.close()

# ------------------------------------------------------------------ final clusterings, compared with known labels
K = 10
km = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Zs)
lab = km.labels_


def ext(pred, name):
    out = {"method": name}
    for target in ("queue", "type", "priority"):
        y = df[target].values
        out[target] = {"ARI": adjusted_rand_score(y, pred), "NMI": normalized_mutual_info_score(y, pred),
                       "homogeneity": homogeneity_score(y, pred), "completeness": completeness_score(y, pred)}
    return out


rows = [ext(lab, "K-Means (k=10) on PCA-50 embeddings")]
rows[0]["silhouette"] = float(silhouette_score(Zs[sample], lab[sample], metric="cosine"))
Xl = normalize(TruncatedSVD(n_components=100, random_state=SEED).fit_transform(X))
km_t = KMeans(n_clusters=K, n_init=20, random_state=SEED).fit(Xl)
rows.append(ext(km_t.labels_, "K-Means (k=10) on TF-IDF + SVD-100"))
rows[-1]["silhouette"] = float(silhouette_score(Xl[sample], km_t.labels_[sample], metric="cosine"))
hs = rng.choice(len(Zs), 3000, replace=False)
agg = AgglomerativeClustering(n_clusters=K, linkage="ward").fit(Zs[hs])
r_agg = {"method": "Agglomerative (Ward, k=10) on 3,000-ticket sample"}
for target in ("queue", "type", "priority"):
    y = df[target].values[hs]
    r_agg[target] = {"ARI": adjusted_rand_score(y, agg.labels_), "NMI": normalized_mutual_info_score(y, agg.labels_),
                     "homogeneity": homogeneity_score(y, agg.labels_), "completeness": completeness_score(y, agg.labels_)}
r_agg["silhouette"] = float(silhouette_score(Zs[hs], agg.labels_, metric="cosine"))
rows.append(r_agg)
R["clustering"] = rows
for r in rows:
    print(r["method"], {t: round(r[t]["NMI"], 3) for t in ("queue", "type", "priority")}, round(r["silhouette"], 3))

# dendrogram on 300 tickets
hs2 = rng.choice(len(Zs), 300, replace=False)
L = linkage(Zs[hs2], method="ward")
fig, ax = plt.subplots(figsize=(10, 3.4))
dendrogram(L, truncate_mode="lastp", p=30, color_threshold=None, above_threshold_color=C["ink"], ax=ax,
           leaf_font_size=8)
ax.set_ylabel("Ward linkage distance"); ax.set_xlabel("Merged groups of tickets (last 30 merges shown)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_5_dendrogram.png"); plt.close()

# cluster profiles: size, dominant queue, top words
terms = np.array(vec.get_feature_names_out())
prof = []
for c in range(K):
    m = lab == c
    q = df.loc[m, "queue"].value_counts()
    mean_tfidf = np.asarray(X[m].mean(0)).ravel() - np.asarray(X[~m].mean(0)).ravel()
    prof.append({"cluster": c, "size": int(m.sum()), "top_queue": q.index[0], "top_queue_share": float(q.iloc[0] / m.sum()),
                 "high_share": float((df.loc[m, "priority"] == "high").mean()),
                 "terms": list(terms[np.argsort(-mean_tfidf)[:6]])})
prof.sort(key=lambda x: -x["size"])
R["cluster_profiles"] = prof
for p in prof:
    print(p)

# t-SNE map of a sample, coloured by K-Means cluster
ts = rng.choice(len(Z), 2500, replace=False)
Y2 = TSNE(n_components=2, perplexity=35, init="pca", random_state=SEED).fit_transform(Z[ts])
fig, ax = plt.subplots(figsize=(7.5, 5.2))
cmap = plt.get_cmap("tab10")
for c in range(K):
    m = lab[ts] == c
    name = next(p for p in prof if p["cluster"] == c)
    ax.scatter(Y2[m, 0], Y2[m, 1], s=6, color=cmap(c), alpha=0.7, label=f"C{c}: {', '.join(name['terms'][:2])}")
ax.set_xticks([]); ax.set_yticks([]); ax.spines[:].set_visible(False)
ax.legend(fontsize=8, markerscale=2.5, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_4_tsne_clusters.png"); plt.close()

# ------------------------------------------------------------------ retrieval (nearest neighbours) from the project evaluation
hk = pd.read_csv("/content/retrieval_hit_at_k.csv")
R["hit_at_k"] = hk.to_dict("records")
sw = pd.read_csv("/content/threshold_sweep.csv")
R["sweep"] = sw.to_dict("records")
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(sw["threshold"], sw["accuracy"], "o-", color=C["ink"], ms=4, label="Overall correct decisions")
ax.plot(sw["threshold"], sw["false_resolve_pct"], "s--", color=C["high"], ms=4, label="Unknown questions wrongly answered")
ax.plot(sw["threshold"], sw["auto_resolve_pct"], "^:", color=C["low"], ms=4, label="Known questions answered")
ax.axvline(0.55, color=C["grey"], ls=":"); ax.text(0.555, 50, "chosen\n0.55", fontsize=9)
ax.set_xlabel("Cosine similarity threshold"); ax.set_ylabel("%"); ax.legend(fontsize=9, frameon=False)
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_6_threshold.png"); plt.close()

# label agreement of near-duplicate tickets (how much the text decides the priority)
best_sim = np.zeros(len(E)); best_j = np.zeros(len(E), dtype=int)
for s in range(0, len(E), 1000):
    S = E[s:s + 1000] @ E.T
    for r in range(S.shape[0]):
        S[r, s + r] = -1
    best_j[s:s + 1000] = S.argmax(1); best_sim[s:s + 1000] = S.max(1)
pr = df["priority"].values
R["near_dup"] = {str(t): {"n": int((best_sim >= t).sum()), "agree": float((pr[best_sim >= t] == pr[best_j[best_sim >= t]]).mean())}
                 for t in (0.95, 0.90, 0.85)}
print(R["near_dup"])
json.dump(R, open(f"{W}/unit3.json", "w"), indent=1, default=float)


# PRAGYAAN – Unit 4
Machine Learning project (21CSC305P). Data: English subset of the Kaggle Multilingual Customer Support Tickets dataset.

In [ ]:
# Upload tickets.parquet and emb.npy (from the zip) to the Colab session first, then run all cells.
!pip -q install xgboost hmmlearn pyarrow
import os; os.makedirs('/content/report_work/fig', exist_ok=True)
!mv -f /content/tickets.parquet /content/emb.npy /content/report_work/ 2>/dev/null; ls /content/report_work

In [ ]:
%%writefile /content/common.py
"""Shared data loading, split and plotting style for the PRAGYAAN report experiments."""
import re
import sys
import time

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402

import re as _re
_PH = _re.compile(r"<(?:tel_num|email|url|account_num|order_num|ip|date|name|address|password|username|phone|amount|id)[^>]*>", _re.I)


def clean_text(text, strict=False):
    out = _re.sub(r"\s+", " ", _PH.sub(" ", text or "")).strip()
    return out.lower() if strict else out

W = "/content/report_work"
FIG = f"{W}/fig"
LABELS = ["high", "medium", "low"]
SEED = 42

plt.rcParams.update({
    "font.family": "Liberation Serif", "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.bbox": "tight",
})
C = {"high": "#B23A48", "medium": "#E0A458", "low": "#4E8098", "ink": "#2B2D42", "grey": "#8D99AE"}


def load():
    df = pd.read_parquet(f"{W}/tickets.parquet")
    for c in ["text", "priority", "type", "queue", "answer"]:
        df[c] = np.array(df[c].tolist(), dtype=object)
    emb = np.load(f"{W}/emb.npy")
    df["clean"] = [clean_text(t, strict=True) for t in df["text"]]
    df["words"] = df["text"].str.split().str.len()
    return df, emb


def split(df):
    idx = np.arange(len(df))
    tr, te = train_test_split(idx, test_size=0.2, stratify=df["priority"], random_state=SEED)
    return tr, te


def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), max_features=2000, stop_words="english", min_df=2)


def evaluate(name, model, Xtr, ytr, Xte, yte, proba=True):
    t0 = time.perf_counter()
    model.fit(Xtr, ytr)
    fit_s = time.perf_counter() - t0
    pred = model.predict(Xte)
    p, r, f, s = precision_recall_fscore_support(yte, pred, labels=LABELS, zero_division=0)
    out = {"model": name, "accuracy": accuracy_score(yte, pred), "macro_f1": f1_score(yte, pred, average="macro"),
           "weighted_f1": f1_score(yte, pred, average="weighted"), "fit_seconds": fit_s,
           "per_class": {lab: {"precision": p[i], "recall": r[i], "f1": f[i], "support": int(s[i])}
                         for i, lab in enumerate(LABELS)}}
    if proba and hasattr(model, "predict_proba"):
        try:
            P = model.predict_proba(Xte)
            out["roc_auc_macro"] = roc_auc_score(yte, P, multi_class="ovr", average="macro", labels=list(model.classes_))
        except Exception:
            pass
    elif hasattr(model, "decision_function"):
        D = model.decision_function(Xte)
        out["roc_auc_macro"] = _auc_from_scores(yte, D, list(model.classes_))
    out["pred"] = list(pred)
    return out


def _auc_from_scores(y, D, classes):
    from scipy.special import softmax
    return roc_auc_score(y, softmax(D, axis=1), multi_class="ovr", average="macro", labels=classes)


def words_re(s):
    return re.findall(r"[a-z]+", s.lower())


In [ ]:
"""Unit 4: sequential modelling of ticket text as word sequences.

Each ticket is an ordered sequence of word symbols. We fit (a) a first-order Markov chain per priority class and
(b) a discrete Hidden Markov Model per priority class (Baum-Welch / forward-backward), classify a test ticket by the
class whose model gives the highest log-likelihood (forward algorithm) plus the log prior, and (c) one shared HMM whose
Viterbi path segments tickets into hidden "discourse" states.
"""
import json
import re
import time
from collections import Counter

import numpy as np
import sys; sys.path.insert(0, '/content')
from common import FIG, LABELS, SEED, C, W, load, plt, split
from hmmlearn.hmm import CategoricalHMM
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support

df, emb = load()
tr, te = split(df)
TOK = re.compile(r"[a-z]+(?:'[a-z]+)?")
STOP_SMALL = {"the", "a", "an", "and", "of", "to", "in", "on", "for", "is", "are", "was", "be", "it", "this", "that",
              "with", "as", "at", "by", "or", "we", "our", "i", "my", "you", "your", "have", "has", "been", "from"}
MAXLEN = 80
V = 600  # vocabulary size (plus UNK)

seqs_txt = [[w for w in TOK.findall(t) if w not in STOP_SMALL][:MAXLEN] for t in df["clean"]]
cnt = Counter(w for i in tr for w in seqs_txt[i])
vocab = [w for w, _ in cnt.most_common(V)]
idx = {w: i for i, w in enumerate(vocab)}
UNK = len(vocab)
seqs = [np.array([idx.get(w, UNK) for w in s] or [UNK]) for s in seqs_txt]
y = df["priority"].to_numpy(dtype=str)
R = {"vocab": V + 1, "maxlen": MAXLEN}
lens = np.array([len(s) for s in seqs])
R["len"] = {"median": float(np.median(lens)), "q1": float(np.percentile(lens, 25)), "q3": float(np.percentile(lens, 75)),
            "max": int(lens.max())}
R["unk_rate"] = float(np.mean([np.mean(seqs[i] == UNK) for i in tr]))
R["tokens_train"] = int(sum(len(seqs[i]) for i in tr))
R["top_tokens"] = cnt.most_common(15)

# hold out 15% of the training part for choosing the number of states
rng = np.random.default_rng(SEED)
tr_perm = rng.permutation(tr)
n_val = int(0.15 * len(tr))
val, fit_idx = tr_perm[:n_val], tr_perm[n_val:]
prior = {c: np.mean(y[tr] == c) for c in LABELS}


def concat(ids):
    X = np.concatenate([seqs[i] for i in ids]).reshape(-1, 1)
    L = [len(seqs[i]) for i in ids]
    return X, L


# ------------------------------------------------------------------ (a) first-order Markov chain per class
def fit_markov(ids, alpha=0.1):
    K = UNK + 1
    start = np.full(K, alpha); trans = np.full((K, K), alpha)
    for i in ids:
        s = seqs[i]; start[s[0]] += 1
        np.add.at(trans, (s[:-1], s[1:]), 1)
    return np.log(start / start.sum()), np.log(trans / trans.sum(1, keepdims=True))


def ll_markov(model, s):
    ls, lt = model
    return ls[s[0]] + lt[s[:-1], s[1:]].sum()


def classify(score_fn, models, ids, use_prior=True):
    out = []
    for i in ids:
        sc = {c: score_fn(models[c], seqs[i]) + (np.log(prior[c]) if use_prior else 0) for c in LABELS}
        out.append(max(sc, key=sc.get))
    return np.array(out)


def scores(pred, ids):
    p, r, f, s = precision_recall_fscore_support(y[ids], pred, labels=LABELS, zero_division=0)
    return {"accuracy": accuracy_score(y[ids], pred), "macro_f1": f1_score(y[ids], pred, average="macro"),
            "per_class": {c: {"precision": p[k], "recall": r[k], "f1": f[k]} for k, c in enumerate(LABELS)}}


alpha_res = {}
for a in (0.01, 0.1, 1.0):
    mm = {c: fit_markov([i for i in fit_idx if y[i] == c], a) for c in LABELS}
    for up in (True, False):
        alpha_res[f"{a}|{up}"] = scores(classify(ll_markov, mm, val, up), val)["macro_f1"]
best_a = max(alpha_res, key=alpha_res.get); R["markov_val"] = alpha_res; R["markov_best"] = best_a
a, up = best_a.split("|"); a = float(a); up = up == "True"
t0 = time.perf_counter()
mm = {c: fit_markov([i for i in tr if y[i] == c], a) for c in LABELS}
mk_fit = time.perf_counter() - t0
pred_mk = classify(ll_markov, mm, te, up)
R["markov_test"] = scores(pred_mk, te); R["markov_test"]["fit_seconds"] = mk_fit
print("markov", best_a, R["markov_test"]["accuracy"], R["markov_test"]["macro_f1"])


# ------------------------------------------------------------------ (b) HMM per class
def fit_hmm(ids, n, iters=40):
    X, L = concat(ids)
    m = CategoricalHMM(n_components=n, n_iter=iters, tol=1e-3, random_state=SEED, init_params="ste", params="ste",
                       n_features=UNK + 1)
    m.fit(X, L)
    return m


def ll_hmm(m, s):
    return m.score(s.reshape(-1, 1))


hmm_val = {}
for n in (2, 4, 8):
    t0 = time.perf_counter()
    hm = {c: fit_hmm([i for i in fit_idx if y[i] == c], n, iters=25) for c in LABELS}
    for up in (True, False):
        hmm_val[f"{n}|{up}"] = scores(classify(ll_hmm, hm, val, up), val)["macro_f1"]
    print("hmm val", n, {k: round(v, 4) for k, v in hmm_val.items() if k.startswith(f"{n}|")}, round(time.perf_counter() - t0, 1))
best_h = max(hmm_val, key=hmm_val.get); R["hmm_val"] = hmm_val; R["hmm_best"] = best_h
n_best, up_h = best_h.split("|"); n_best = int(n_best); up_h = up_h == "True"
t0 = time.perf_counter()
hm = {c: fit_hmm([i for i in tr if y[i] == c], n_best, iters=40) for c in LABELS}
hm_fit = time.perf_counter() - t0
pred_h = classify(ll_hmm, hm, te, up_h)
R["hmm_test"] = scores(pred_h, te); R["hmm_test"]["fit_seconds"] = hm_fit
R["hmm_test_cm"] = confusion_matrix(y[te], pred_h, labels=LABELS).tolist()
R["hmm_converged"] = {c: bool(hm[c].monitor_.converged) for c in LABELS}
R["hmm_iters"] = {c: int(hm[c].monitor_.iter) for c in LABELS}
print("hmm", best_h, R["hmm_test"]["accuracy"], R["hmm_test"]["macro_f1"])

# unigram baseline on the same symbols (order ignored) for a fair comparison
def fit_uni(ids, a=0.1):
    c = np.full(UNK + 1, a)
    for i in ids:
        np.add.at(c, seqs[i], 1)
    return np.log(c / c.sum())


um = {c: fit_uni([i for i in tr if y[i] == c]) for c in LABELS}
pred_u = classify(lambda m, s: m[s].sum(), um, te, up)
R["unigram_test"] = scores(pred_u, te)
print("unigram", R["unigram_test"]["macro_f1"])

# ------------------------------------------------------------------ (c) shared HMM for discourse states + Viterbi
NS = 5
X, L = concat(tr)
g = CategoricalHMM(n_components=NS, n_iter=60, tol=1e-3, random_state=SEED, n_features=UNK + 1).fit(X, L)
R["global_ll_per_token"] = float(g.score(X, L) / len(X))
R["global_converged"] = bool(g.monitor_.converged); R["global_iters"] = int(g.monitor_.iter)
inv = np.array(vocab + ["<unk>"])
states = []
for s_ in range(NS):
    em = g.emissionprob_[s_].copy(); em[UNK] = 0
    states.append({"state": s_, "top": list(inv[np.argsort(-em)[:8]]), "start": float(g.startprob_[s_]),
                   "stay": float(g.transmat_[s_, s_])})
R["global_states"] = states
R["global_trans"] = g.transmat_.round(3).tolist()
R["global_start"] = g.startprob_.round(3).tolist()
# share of tokens in each state, and position profile
occ = np.zeros((NS, 10))
for i in te[:800]:
    _, path = g.decode(seqs[i].reshape(-1, 1), algorithm="viterbi")
    pos = np.minimum((np.arange(len(path)) / max(len(path), 1) * 10).astype(int), 9)
    np.add.at(occ, (path, pos), 1)
occ_norm = occ / occ.sum(0, keepdims=True)
R["state_position"] = occ_norm.round(3).tolist()
# a decoded example
ex_i = te[3]
_, path = g.decode(seqs[ex_i].reshape(-1, 1), algorithm="viterbi")
R["example"] = {"words": seqs_txt[ex_i][:40], "states": path[:40].tolist(), "priority": y[ex_i]}

# figures
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1.25]})
im = ax[0].imshow(g.transmat_, cmap="Greys", vmin=0, vmax=1)
for i in range(NS):
    for j in range(NS):
        v = g.transmat_[i, j]
        ax[0].text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.5 else "black", fontsize=9)
ax[0].set_xticks(range(NS)); ax[0].set_yticks(range(NS)); ax[0].set_xticklabels([f"S{i}" for i in range(NS)])
ax[0].set_yticklabels([f"S{i}" for i in range(NS)]); ax[0].set_xlabel("To state"); ax[0].set_ylabel("From state")
ax[0].set_title("(a) Transition matrix A"); ax[0].spines[:].set_visible(False)
cmap = plt.get_cmap("tab10")
bottom = np.zeros(10)
for s_ in range(NS):
    ax[1].bar(np.arange(10) * 10 + 5, occ_norm[s_] * 100, width=9, bottom=bottom, color=cmap(s_),
              label=f"S{s_}: {', '.join(states[s_]['top'][:3])}")
    bottom += occ_norm[s_] * 100
ax[1].set_xlabel("Position in ticket (% of length)"); ax[1].set_ylabel("Share of words (%)")
ax[1].set_title("(b) Viterbi state by position (800 test tickets)")
ax[1].legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_9_hmm_states.png"); plt.close()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].hist(lens, bins=40, color=C["ink"])
ax[0].set_xlabel("Sequence length (word symbols, capped at 80)"); ax[0].set_ylabel("Tickets"); ax[0].set_title("(a) Sequence lengths")
freqs = np.array(sorted(cnt.values(), reverse=True))
ax[1].loglog(np.arange(1, len(freqs) + 1), freqs, color=C["ink"])
ax[1].axvline(V, ls=":", color=C["high"]); ax[1].text(V * 1.1, freqs[0] / 3, f"vocabulary cut\n({V} words)", fontsize=9)
ax[1].set_xlabel("Word rank"); ax[1].set_ylabel("Frequency (training part)"); ax[1].set_title("(b) Word frequency (Zipf plot)")
plt.tight_layout(); plt.savefig(f"{FIG}/fig3_8_seq_eda.png"); plt.close()

json.dump(R, open(f"{W}/unit4.json", "w"), indent=1, default=float)
print(json.dumps({k: R[k] for k in ("markov_best", "hmm_best", "hmm_converged", "global_states")}, default=float)[:1500])
